In [1]:
import os

from astropy.io import fits
from astropy import units as u
from astropy.coordinates import Angle

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.6f}'.format)

In [2]:
def format_ra_dec(row):
    ra_angle = Angle(f"{star['ra_hours']}h{star['ra_minutes']}m{star['ra_seconds']}s")
    ra = ra_angle.to_string(unit=u.hour, sep=' ', precision=2, pad=True)

    dec_angle = Angle(f"{star['dec_degrees']}d{star['dec_minutes']}m{star['dec_seconds']}s")
    dec = dec_angle.to_string(unit=u.degree, sep=' ', precision=2, alwayssign=True, pad=True)

    return ra, dec

def dms_to_decimal(degrees, minutes, seconds, direction):
    return Angle(f"{degrees}d{minutes}m{seconds}s{direction}").value

def dump_header(filename):
    with fits.open(filename) as hdul:
        for hdu in hdul:
            print("Header of HDU:", hdu.name)
            for key, value in hdu.header.items():
                print(f"{key}: {value}")
            print("\n")

def update_header(filename, values_to_add, keys_to_remove):
    with fits.open(filename, mode='update') as hdul:

        for key, value in values_to_add.items():
            hdul[0].header[key] = value

        for key in keys_to_remove:
            if key in hdul[0].header:
                del hdul[0].header[key]
        hdul.flush()

In [3]:
df = pd.read_csv('Variable Stars.csv')
df[0:100]

,friendlyName,ra_hours,ra_minutes,ra_seconds,dec_degrees,dec_minutes,dec_seconds,magnitudeHigh,magnitudeLow,period,constellation,type,spectralType
0,W UMa,9,43,45.470000,55,57,9.100000,7.750000,8.480000,0.333637,Ursa Major,EW/KW,F8Vp+F8Vp
1,Z UMa,11,56,30.220000,57,52,17.600000,6.200000,9.400000,195.500000,Ursa Major,SRB,M5IIIe
2,del Cep,22,29,10.270000,58,24,54.700000,3.490000,4.360000,5.366266,Cepheus,DCEP,F5Ib-G1Ib
3,bet Per,3,8,10.130000,40,57,20.400000,2.090000,3.300000,2.867350,Perseus,EA/SD,B8V+G8III
4,bet Lyr,18,50,4.790000,33,21,45.600000,3.300000,4.350000,12.944000,Lyra,DPV:/EB,B8II-IIIep
5,mu Cep,21,43,30.460000,58,46,48.100000,3.430000,5.100000,835.000000,Cepheus,SRC,M2Iae
6,VW Cep,20,37,21.540000,75,36,1.500000,7.310000,7.710000,0.278309,Cepheus,EW/KW,G5+K0Ve
7,eta Aql,19,52,28.370000,1,0,20.400000,3.490000,4.300000,7.176790,Aquila,DCEP,F6Ib-G4Ib+B9.8V
8,omi Cet,2,19,20.790000,-2,58,39.500000,2.000000,10.100000,330.200000,Cetus,M,M5e-M9e
9,R Lyr,18,55,20.100000,43,56,45.400000,3.810000,4.440000,46.000000,Lyra,SRB,M5III


In [4]:
obj = "V0338 Boo"
star = df.loc[df['friendlyName'] == obj].iloc[0]  # Get the first matching row
objectra, objectdec = format_ra_dec(star)

latitude = dms_to_decimal(19, 42, 29.97, 'N')
longitude = dms_to_decimal(155, 58, 45.07, 'W')
altitude = (1535.4 * u.imperial.ft).to(u.m).value

print("Object:", obj)
print("RA:", objectra)
print("Dec:", objectdec)

print("latitude:", latitude)
print("longitude:", longitude)
print("altitude:", altitude)


Object: V0338 Boo
RA: 15 46 26.14
Dec: +44 18 47.20
latitude: 19.708325
longitude: -155.97918611111112
altitude: 467.98992000000004


In [5]:
directory = '/Users/jwatts/astrophotography/Photometry/V0338Boo/'

values_to_add = {
    'SITEELEV': (altitude, 'Site elevation in meters'),
    'SITELAT': (latitude, 'Site latitude in degrees'),
    'SITELONG': (longitude, 'Site longitude in degrees'),
    'OBJECT': (obj, 'Target object name'),
    'OBJCTRA': (objectra, '[hms J2000] Target right ascension'),
    'OBJCTDEC': (objectdec, '[dms +N J2000] Target declination')
}

keys_to_remove = ['RA', 'DEC']

for filename in os.listdir(directory):
    if filename.endswith('.fit'):
        fits_file = os.path.join(directory, filename)
        print(f"FITS FILE: {fits_file}")
        update_header(fits_file, values_to_add, keys_to_remove)
        dump_header(fits_file)


FITS FILE: /Users/jwatts/astrophotography/Photometry/V0338Boo/Light_V0338Boo_20.0s_Bin1_20240416-222727_0008.fit
Header of HDU: PRIMARY
SIMPLE: True
BITPIX: 16
NAXIS: 2
NAXIS1: 4144
NAXIS2: 2822
EXTEND: True
COMMENT:   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT:   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H
BZERO: 32768
BSCALE: 1
CREATOR: ZWO ASIAIR Plus
OFFSET: 30
XORGSUBF: 0
YORGSUBF: 0
FOCALLEN: 1285
SET-TEMP: -20
EGAIN: 3.99000000953674
XBINNING: 1
YBINNING: 1
CCDXBIN: 1
CCDYBIN: 1
XPIXSZ: 4.63000011444092
YPIXSZ: 4.63000011444092
IMAGETYP: Light
EXPOSURE: 20.0
EXPTIME: 20.0
CCD-TEMP: -19.0
DATE-OBS: 2024-04-17T08:27:06.282727
INSTRUME: ZWO ASI294MC Pro
BAYERPAT: RGGB
GAIN: 0
TELESCOP: Celestron AVX/CGE/CGEM/CGX
CTYPE1: RA---TAN-SIP
CTYPE2: DEC--TAN-SIP
CRVAL1: 236.489883546
CRVAL2: 44.0847473857
CRPIX1: 961.257781984
CRPIX2: 1987.960876464
CD1_1: -2.0266344837925e-05
CD1_2: -0.0002059333092865
CD2_1: 0.00020541431119825